In [ ]:
"""
bike_route.py

Fetches the best cycling route between two locations using the
Google Maps Directions API and prints a turn-by-turn summary.

Usage
-----
    python3 bike_route.py

Or import and call get_bike_route() from another script.

.env file (place in the same directory as this script)
-------------------------------------------------------
    GOOGLE_MAPS_API_KEY=your_key_here

Dependencies
------------
    pip install googlemaps python-dotenv
"""

import os
from pathlib import Path

import googlemaps
from dotenv import load_dotenv

# ---------------------------------------------------------------------------
# Load API key from .env
# ---------------------------------------------------------------------------
load_dotenv(Path(__file__).parent / ".env")
API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")

if not API_KEY:
    raise EnvironmentError(
        "GOOGLE_MAPS_API_KEY not found. "
        "Make sure it is set in Google_Maps_Data/.env"
    )

# ---------------------------------------------------------------------------
# Core function
# ---------------------------------------------------------------------------

def get_bike_route(origin: str, destination: str) -> dict:
    """
    Returns the best cycling route from origin to destination.

    Parameters
    ----------
    origin      : str  – address, landmark, or "lat,lng" string
    destination : str  – address, landmark, or "lat,lng" string

    Returns
    -------
    dict with keys:
        summary        – short route description (e.g. "Main St and Broadway")
        distance_km    – float, total distance in kilometres
        duration_min   – float, estimated cycling time in minutes
        steps          – list of dicts, each with:
                           instruction  (plain-text turn-by-turn)
                           distance_km
                           duration_min
                           start_lat, start_lng
                           end_lat,   end_lng
        polyline       – encoded polyline string for the whole route
        warnings       – list of strings (Google sometimes adds cycling warnings)
        raw_leg        – the full raw leg dict from the API response
    """
    gmaps = googlemaps.Client(key=API_KEY)

    result = gmaps.directions(
        origin,
        destination,
        mode="bicycling",       # cycling-optimised route
        alternatives=False,     # set True to get multiple route options
        units="metric",
    )

    if not result:
        raise ValueError(
            f"No cycling route found between '{origin}' and '{destination}'. "
            "Check that both addresses are valid and reachable by bike."
        )

    route = result[0]                        # best route
    leg   = route["legs"][0]                 # single leg (no waypoints)

    # ---- Parse steps -------------------------------------------------------
    steps = []
    for step in leg["steps"]:
        # Strip HTML tags from Google's instruction strings
        import re
        instruction = re.sub(r"<[^>]+>", "", step.get("html_instructions", ""))

        steps.append({
            "instruction":  instruction,
            "distance_km":  step["distance"]["value"] / 1000,
            "duration_min": step["duration"]["value"] / 60,
            "start_lat":    step["start_location"]["lat"],
            "start_lng":    step["start_location"]["lng"],
            "end_lat":      step["end_location"]["lat"],
            "end_lng":      step["end_location"]["lng"],
        })

    return {
        "summary":      route.get("summary", ""),
        "distance_km":  leg["distance"]["value"] / 1000,
        "duration_min": leg["duration"]["value"] / 60,
        "steps":        steps,
        "polyline":     route["overview_polyline"]["points"],
        "warnings":     route.get("warnings", []),
        "raw_leg":      leg,
    }


# ---------------------------------------------------------------------------
# Pretty-print helper
# ---------------------------------------------------------------------------

def print_route(route: dict) -> None:
    """Prints a human-readable turn-by-turn summary."""
    print(f"\n{'='*60}")
    print(f"  Route summary : {route['summary']}")
    print(f"  Distance      : {route['distance_km']:.2f} km")
    print(f"  Duration      : {route['duration_min']:.0f} min")
    if route["warnings"]:
        for w in route["warnings"]:
            print(f"  ⚠ Warning      : {w}")
    print(f"{'='*60}")

    for i, step in enumerate(route["steps"], start=1):
        print(
            f"  {i:>2}. {step['instruction']}"
            f"  ({step['distance_km']:.2f} km, ~{step['duration_min']:.0f} min)"
        )

    print(f"{'='*60}\n")


# ---------------------------------------------------------------------------
# Quick demo — edit the addresses below to test
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    START = "Boston City Hall, Boston, MA"
    END   = "Fenway Park, Boston, MA"

    print(f"Finding bike route:\n  From : {START}\n  To   : {END}\n")

    route = get_bike_route(START, END)
    print_route(route)

    # The encoded polyline can be decoded with googlemaps.convert.decode_polyline()
    # or pasted into https://developers.google.com/maps/documentation/utilities/polylineutility
    print(f"Encoded polyline (paste into Maps Polyline Utility to visualise):")
    print(f"  {route['polyline'][:80]}…")
